In [41]:
# ==============================================================================
# CELL 1: SETUP & LOAD GEOJSON
# ==============================================================================
import geopandas as gpd
import pandas as pd
from google.colab import drive
import os

print("⏳ Mounting Google Drive...")
try:
    drive.mount('/content/drive')
    print("✅ Drive Mounted.")
except:
    print("ℹ️ Drive already mounted or running locally.")

# Define paths (Adjust if your folder structure is slightly different)
INPUT_GEOJSON = '/content/drive/MyDrive/_Pienza/Assets/Phase_2/poly.geojson'
OUTPUT_GEOJSON = '/content/drive/MyDrive/_Pienza/Assets/Phase_2/poly_numbered_streamlit.geojson'
OUTPUT_CSV = '/content/drive/MyDrive/_Pienza/Assets/Phase_2/polygon_latex_index_streamlit.csv'

# Load the raw polygons
try:
    gdf = gpd.read_file(INPUT_GEOJSON)

    # Standardize the name column based on your previous notebook's logic
    if 'Name' in gdf.columns:
        gdf = gdf.rename(columns={'Name': 'zone_name'})
    elif 'name' in gdf.columns:
        gdf = gdf.rename(columns={'name': 'zone_name'})

    print(f"✅ Successfully loaded {len(gdf)} polygons.")
except Exception as e:
    print(f"🔴 ERROR loading GeoJSON: {e}")

⏳ Mounting Google Drive...
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ Drive Mounted.
✅ Successfully loaded 72 polygons.


In [42]:
# ==============================================================================
# CELL 2: INJECT IDs & GENERATE ARTIFACTS
# ==============================================================================
from IPython.display import display, Markdown

# 1. Sort alphabetically so the final LaTeX table is easy for the reader to scan
gdf = gdf.sort_values('zone_name').reset_index(drop=True)

# 2. Inject the numerical ID (starting at 1)
gdf['map_id'] = gdf.index + 1

# 3. Export the rich GeoJSON for Kepler.gl
gdf.to_file(OUTPUT_GEOJSON, driver='GeoJSON')
display(Markdown(f"🗺️ **Numbered GeoJSON saved:** `{OUTPUT_GEOJSON}`"))
display(Markdown("*Drag and drop this file directly into the Kepler.gl web UI.*"))

# 4. Extract and export the clean index for your LaTeX table
df_index = gdf[['map_id', 'zone_name']]
df_index.to_csv(OUTPUT_CSV, index=False)
display(Markdown(f"📊 **LaTeX Reference CSV saved:** `{OUTPUT_CSV}`"))

# Preview the mapping
display(df_index.head(10))

🗺️ **Numbered GeoJSON saved:** `/content/drive/MyDrive/_Pienza/Assets/Phase_2/poly_numbered_streamlit.geojson`

*Drag and drop this file directly into the Kepler.gl web UI.*

📊 **LaTeX Reference CSV saved:** `/content/drive/MyDrive/_Pienza/Assets/Phase_2/polygon_latex_index_streamlit.csv`

,map_id,zone_name
0,1,agwa_bezares
1,2,ahuehuetes_norte
2,3,ahuehuetes_sur
3,4,anahuac_1
4,5,anzures
5,6,ave_club_de_golf_lomas
6,7,bahias
7,8,blvrd_anahuac
8,9,bondojito_asf
9,10,bosque_1


In [43]:
import pandas as pd
import geopandas as gpd

# 1. Temporarily project to a flat CRS (Mexico City is roughly UTM Zone 14N -> EPSG:32614)
# (Or we can use Web Mercator EPSG:3857 which works universally for flat geometry)
gdf_flat = gdf.to_crs(epsg=3857)

# 2. Calculate the mathematically perfect centroid on the flat projection
centroids_flat = gdf_flat.geometry.centroid

# 3. Convert JUST those center points back to Lat/Lon (EPSG:4326) for Kepler
centroids_latlon = centroids_flat.to_crs(epsg=4326)

# 4. Extract the coordinates and your map_id
df_labels = pd.DataFrame({
    'map_id': gdf['map_id'],
    'zone_name': gdf['zone_name'],
    'lat': centroids_latlon.y,
    'lon': centroids_latlon.x
})

# 5. Export the clean, perfectly centered labels
OUTPUT_LABELS = '/content/drive/MyDrive/_Pienza/Assets/Phase_2/polygon_labels_streamlit.csv'
df_labels.to_csv(OUTPUT_LABELS, index=False)

print(f"✅ Mathematically perfect centroids calculated and saved to: {OUTPUT_LABELS}")

✅ Mathematically perfect centroids calculated and saved to: /content/drive/MyDrive/_Pienza/Assets/Phase_2/polygon_labels_streamlit.csv


In [44]:
import geopandas as gpd
import pandas as pd
from IPython.display import display, Markdown
import copy

# 1. Load your original GeoJSON
INPUT_GEOJSON = '/content/drive/MyDrive/_Pienza/Assets/Phase_2/poly.geojson'
OUTPUT_GEOJSON = '/content/drive/MyDrive/_Pienza/Assets/Phase_2/poly_numbered_streamlit.geojson'

gdf = gpd.read_file(INPUT_GEOJSON)

if 'Name' in gdf.columns:
    gdf = gdf.rename(columns={'Name': 'zone_name'})
elif 'name' in gdf.columns:
    gdf = gdf.rename(columns={'name': 'zone_name'})

# 2. The Duplicate-Proof Dictionary
# Notice how every value is a LIST. For tecamachalco, we give it BOTH IDs.
zone_to_ids = {
    'anzures': [1], 'anahuac_1': [2], 'bahias': [3], 'rios': [4], 'juarez_rosa': [5],
    'juarez_soho_house': [6], 'roma_condesa_2': [7], 'roma_condesa_1': [8], 'bosque_1': [9],
    'campo_marte': [10], 'polanco_parque_lincoln': [11], 'polanco_gandhi': [12], 'polanco_5': [13],
    'polanco_parroquia': [14], 'lagos': [15], 'frontera_polanco': [16], 'carso_antara_miyana': [17],
    'polanco_palacio': [18], 'polanco_grupo_mexico': [19], 'polanco_uber_hq': [20], 'irrigacion': [21],
    'sotelo': [22], 'sedena': [23],
    'tecamachalco': [24, 38], # <-- THE FIX: It will assign 24 first, then 38 to the next one
    'fuentes_casino': [25], 'lomas_barrilaco': [26], 'lomas_olimpo': [27], 'lomas_prado_norte': [28],
    'palmas_jp_morgan': [29], 'lomas_fc_cuernavaca': [30], 'lomas_virreyes': [31], 'bosque_2': [32],
    'bondojito_asf': [33], 'bosque_3': [34], 'lomas_trastevere': [35], 'nodo_reforma_palmas': [36],
    'nodo_monte_libano': [37], 'de_las_fuentes': [39], 'herradura_conscripto': [40],
    'de_los_bosques': [41], 'ahuehuetes_norte': [42], 'ahuehuetes_sur': [43], 'reforma_regina': [44],
    'lomas_altas': [45], 'reforma_bnp': [46], 'agwa_bezares': [47], 'sante_fe_patio': [48],
    'santa_fe_quintana': [49], 'santa_fe_tec': [50], 'santa_fe_cumbres_de': [51], 'santa_fe_bosques_de': [52],
    'santa_fe_colegios': [53], 'santa_fe_ibero': [54], 'santa_fe_centro_comercial': [55], 'cruce_echanove': [56],
    'carretera_libre': [57], 'vistahermosa': [58], 'tamarindos': [59], 'bosques_pabellon': [60],
    'loma_de_la_palma': [61], 'carretera_al_olivo': [62], 'vialidad_de_la_barranca': [63], 'el_olivo': [64],
    'blvrd_anahuac': [65], 'universidad_anahuac': [66], 'interlomas_magnocentro': [67], 'ave_club_de_golf_lomas': [68],
    'interlomas_haciendas': [69], 'jesus_del_monte': [70], 'lomas_country_club': [71], 'bosque_real': [72]
}

# 3. Smart Assignment Function
tracker = copy.deepcopy(zone_to_ids)

def assign_id(name):
    if name in tracker and len(tracker[name]) > 0:
        return tracker[name].pop(0) # Gives the first ID in the list, then removes it
    return None

gdf['map_id'] = gdf['zone_name'].apply(assign_id)

# 4. Save the updated GeoJSON
gdf = gdf.sort_values('map_id').reset_index(drop=True)
gdf.to_file(OUTPUT_GEOJSON, driver='GeoJSON')

# 5. Regenerate the Centroids for the visible Kepler numbers!
gdf_flat = gdf.to_crs(epsg=3857)
centroids_latlon = gdf_flat.geometry.centroid.to_crs(epsg=4326)

df_labels = pd.DataFrame({
    'map_id': gdf['map_id'],
    'zone_name': gdf['zone_name'],
    'lat': centroids_latlon.y,
    'lon': centroids_latlon.x
})

OUTPUT_LABELS = '/content/drive/MyDrive/_Pienza/Assets/Phase_2/polygon_labels_streamlit.csv'
df_labels.to_csv(OUTPUT_LABELS, index=False)

display(Markdown("✅ **FIXED! Both GeoJSON and Labels CSV updated with distinct Tecamachalco IDs.**"))

✅ **FIXED! Both GeoJSON and Labels CSV updated with distinct Tecamachalco IDs.**

In [45]:
# 5. Extraer y exportar el índice limpio (Solo ID y Nombre)
df_index = gdf[['map_id', 'zone_name']].copy()

# Asegurarnos de que el ID sea entero (integer) para que se vea limpio
df_index['map_id'] = df_index['map_id'].astype(int)

OUTPUT_CSV = '/content/drive/MyDrive/_Pienza/Assets/Phase_2/polygon_latex_index_streamlit.csv'
df_index.to_csv(OUTPUT_CSV, index=False)

display(Markdown(f"📊 **CSV de Referencia guardado en:** `{OUTPUT_CSV}`"))

# Un pequeño preview para confirmar
display(df_index.head(10))

📊 **CSV de Referencia guardado en:** `/content/drive/MyDrive/_Pienza/Assets/Phase_2/polygon_latex_index_streamlit.csv`

,map_id,zone_name
0,1,anzures
1,2,anahuac_1
2,3,bahias
3,4,rios
4,5,juarez_rosa
5,6,juarez_soho_house
6,7,roma_condesa_2
7,8,roma_condesa_1
8,9,bosque_1
9,10,campo_marte


In [46]:
# 1. Temporarily project to flat CRS for perfect centers
gdf_flat = gdf.to_crs(epsg=3857)

# 2. Get the centroids and project back to Lat/Lon
centroids_latlon = gdf_flat.geometry.centroid.to_crs(epsg=4326)

# 3. Create the updated labels dataframe (Now with the NEW map_ids!)
df_labels = pd.DataFrame({
    'map_id': gdf['map_id'],
    'zone_name': gdf['zone_name'],
    'lat': centroids_latlon.y,
    'lon': centroids_latlon.x
})

# 4. Export the updated labels
OUTPUT_LABELS = '/content/drive/MyDrive/_Pienza/Assets/Phase_2/polygon_labels_streamlit.csv'
df_labels.to_csv(OUTPUT_LABELS, index=False)

In [47]:
# ==============================================================================
# CELL: OPUS DB CONNECTIVITY + HDBSCAN CON JITTER Y EXPORTACIÓN A KEPLER
# ==============================================================================
# !pip install hdbscan

import os
import hdbscan
import plotly.express as px
import pandas as pd
import numpy as np
from sqlalchemy import create_engine
from sklearn.preprocessing import StandardScaler
from IPython.display import display, Markdown

display(Markdown("### 🤖 **Invocando a la Bestia: HDBSCAN (Directamente desde OPUS DB)**"))

# --- 0. BOOTSTRAP: CONEXIÓN A OPUS DB ---
DB_PATH = '/content/drive/MyDrive/_Pienza/Assets/Database/opus.db'

if not os.path.exists(DB_PATH):
    raise FileNotFoundError(f"🔴 CRÍTICO: No se encuentra la base de datos en {DB_PATH}. ¿Está montado Google Drive?")
else:
    db_engine = create_engine(f'sqlite:///{DB_PATH}')
    print("✅ Motor SQL Conectado a OPUS DB.")

# --- 1. RECONSTRUCCIÓN DE DATOS ---
query_geo = "SELECT offer_id, dropoff_lat, dropoff_lon FROM offers WHERE dropoff_lat IS NOT NULL AND dropoff_lon IS NOT NULL"
df_hdbscan_analysis = pd.read_sql(query_geo, db_engine)

coords = df_hdbscan_analysis[['dropoff_lat', 'dropoff_lon']]
scaler = StandardScaler()
coords_scaled = scaler.fit_transform(coords)
print(f"✅ Datos extraídos de OPUS DB. Shape: {coords_scaled.shape}")

# --- 2. EJECUCIÓN DE HDBSCAN ---
clusterer = hdbscan.HDBSCAN(min_cluster_size=25, min_samples=5, cluster_selection_epsilon=0.00)
print("⏳ HDBSCAN está analizando topología...")
cluster_labels = clusterer.fit_predict(coords_scaled)
df_hdbscan_analysis['hdbscan_cluster_id'] = cluster_labels
print("✅ Clustering HDBSCAN completado.")

# --- 3. ANÁLISIS DE RESULTADOS ---
n_clusters = len(set(cluster_labels)) - (1 if -1 in cluster_labels else 0)
n_noise = list(cluster_labels).count(-1)
display(Markdown(f"### 🗺️ **Resultados:** {n_clusters} Clusters + {n_noise} Puntos de Ruido"))

# --- 4. DOCTRINA "JITTER" ---
np.random.seed(42)
noise_level = 0.0008
df_hdbscan_analysis['lat_viz'] = df_hdbscan_analysis['dropoff_lat'] + np.random.normal(0, noise_level, len(df_hdbscan_analysis))
df_hdbscan_analysis['lon_viz'] = df_hdbscan_analysis['dropoff_lon'] + np.random.normal(0, noise_level, len(df_hdbscan_analysis))
print(f"✅ Doctrina 'Jitter' aplicada.")

# Convertir a string para forzar variables categóricas
df_hdbscan_analysis['cluster_id_str'] = df_hdbscan_analysis['hdbscan_cluster_id'].astype(str)

# --- 5. EXPORTACIÓN PARA KEPLER.GL ---
OUTPUT_HDBSCAN = '/content/drive/MyDrive/_Pienza/Assets/Phase_2/hdbscan_clusters_streamlit.csv'
df_export = df_hdbscan_analysis[['offer_id', 'lat_viz', 'lon_viz', 'cluster_id_str']].copy()
df_export.to_csv(OUTPUT_HDBSCAN, index=False)
print(f"✅ Archivo para Kepler exportado a: {OUTPUT_HDBSCAN}")

# --- 6. VISUALIZACIÓN IN-NOTEBOOK (PLOTLY) ---
fig = px.scatter_mapbox(
    df_hdbscan_analysis,
    lat="lat_viz",
    lon="lon_viz",
    color="cluster_id_str",
    mapbox_style="carto-positron",
    zoom=10.5, height=900,
    title=f'Mapa de Clusters HDBSCAN: {n_clusters} Zonas Densas',
    color_discrete_sequence=px.colors.qualitative.Vivid,
    color_discrete_map={'-1': 'lightgrey'}, # Ruido al fondo
    category_orders={'cluster_id_str': sorted(df_hdbscan_analysis['cluster_id_str'].unique(), key=lambda x: int(x))}
)
fig.show()

### 🤖 **Invocando a la Bestia: HDBSCAN (Directamente desde OPUS DB)**

✅ Motor SQL Conectado a OPUS DB.
✅ Datos extraídos de OPUS DB. Shape: (4760, 2)
⏳ HDBSCAN está analizando topología...
✅ Clustering HDBSCAN completado.


### 🗺️ **Resultados:** 45 Clusters + 2144 Puntos de Ruido

✅ Doctrina 'Jitter' aplicada.
✅ Archivo para Kepler exportado a: /content/drive/MyDrive/_Pienza/Assets/Phase_2/hdbscan_clusters_streamlit.csv


In [48]:
# ==============================================================================
# CELL: BAUTIZO DEFINITIVO (FINAL COMMIT) CON EXPORTACIÓN
# ==============================================================================
import pandas as pd
from IPython.display import display, Markdown

display(Markdown("### 🏛️ **Aplicando Nomenclatura Final (Bautizo Definitivo)**"))

# 1. DICCIONARIO MAESTRO OFICIAL
zone_naming_map = {
    -1: 'unassigned',
    0: 'viaducto_tlalpan',
    1: 'terminal_1_aicm',
    2: 'central_del_norte',
    3: 'terminal_2_aicm',       # Fusión
    4: 'terminal_2_aicm',       # Fusión
    5: 'plaza_satelite_mundo_e',
    6: 'lomas_verdes',
    7: 'pedregal',
    8: 'san_jeronimo_tizapan',
    9: 'san_angel_altavista',
    10: 'observatorio',
    11: 'sotelo_san_esteban',
    12: 'barranca_del_muerto',
    13: 'haciendas_san_fernando',
    14: 'vialidad_de_la_barranca',
    15: 'interlomas_magnocentro',
    16: 'santa_fe_itesm',
    17: 'cuajimalpa_centro',
    18: 'santa_fe_patio',
    19: 'santa_fe_abc',
    20: 'tamarindos',
    21: 'nodo_constituyentes_reforma',
    22: 'felix_cuevas',
    23: 'cruce_echanove',
    24: 'santa_fe_core',
    25: 'pabellon_bosques',
    26: 'duraznos',
    27: 'chamizal',
    28: 'lomas_anahuac',
    29: 'centro_historico',
    30: 'napoles_wtc',
    31: 'san_miguel_chapultepec',
    32: 'tacubaya',
    33: 'cofre_de_perote',
    34: 'condesa',
    35: 'roma_norte',
    36: 'col_juarez',
    37: 'la_diana',
    38: 'anzures_reforma',
    39: 'polanco_trebol',
    40: 'carso_antara_miyana',
    41: 'el_semaforo_de_palmas',
    42: 'campos_eliseos',
    43: 'virreyes',
    44: 'fc_cuernavaca'
}

# 2. APLICAR MAPEO
# Aseguramos que el ID sea entero para evitar errores de llave
df_hdbscan_analysis['hdbscan_cluster_id'] = df_hdbscan_analysis['hdbscan_cluster_id'].astype(int)

print(f"🔄 Bautizando {len(df_hdbscan_analysis)} registros...")
df_hdbscan_analysis['zone_name'] = df_hdbscan_analysis['hdbscan_cluster_id'].map(zone_naming_map)

# 3. SAFETY CHECK (Limpieza de Huérfanos)
# Si quedó algún ID que no estaba en tu lista (ej. un cluster nuevo sorpresa), lo marcamos
df_hdbscan_analysis['zone_name'] = df_hdbscan_analysis['zone_name'].fillna('ERROR_SIN_BAUTIZO')

# 4. CONFIRMACIÓN VISUAL
display(Markdown("#### **✅ Estado Final del Territorio**"))
conteo = df_hdbscan_analysis['zone_name'].value_counts().reset_index()
conteo.columns = ['Zona Bautizada', 'Total Ofertas']
display(conteo.head(15).style.background_gradient(cmap='Greens'))

# Verificación rápida de la fusión T2
t2_count = len(df_hdbscan_analysis[df_hdbscan_analysis['zone_name'] == 'terminal_2_aicm'])
print(f"\n✈️ Terminal 2 AICM consolidada con {t2_count} viajes.")

# 5. EXPORTACIÓN A GOOGLE DRIVE
OUTPUT_CSV = '/content/drive/MyDrive/_Pienza/Assets/Phase_2/hdbscan_bautizados_final_streamlit.csv'

# Guardamos el DataFrame resultante
df_hdbscan_analysis.to_csv(OUTPUT_CSV, index=False)
display(Markdown(f"\n💾 **Archivo maestro con nomenclaturas guardado exitosamente en:** `{OUTPUT_CSV}`"))

### 🏛️ **Aplicando Nomenclatura Final (Bautizo Definitivo)**

🔄 Bautizando 4760 registros...


#### **✅ Estado Final del Territorio**

,Zona Bautizada,Total Ofertas
0,unassigned,2144
1,santa_fe_core,331
2,carso_antara_miyana,170
3,terminal_1_aicm,111
4,terminal_2_aicm,111
5,sotelo_san_esteban,105
6,el_semaforo_de_palmas,96
7,vialidad_de_la_barranca,86
8,nodo_constituyentes_reforma,85
9,campos_eliseos,75



✈️ Terminal 2 AICM consolidada con 111 viajes.



💾 **Archivo maestro con nomenclaturas guardado exitosamente en:** `/content/drive/MyDrive/_Pienza/Assets/Phase_2/hdbscan_bautizados_final_streamlit.csv`

In [49]:
# ==============================================================================
# CELL: EXPORTACIÓN PRÍSTINA PARA KEPLER (SIN RUIDO / UNASSIGNED)
# ==============================================================================
import pandas as pd
from IPython.display import display, Markdown

display(Markdown("### 🧹 **Operación Escoba: Limpiando el Mapa para Kepler.gl**"))

# --- 1. FILTRADO ESTRICTO ---
# Nos deshacemos de los 'unassigned' y cualquier posible huérfano
df_kepler_clean = df_hdbscan_analysis[
    ~df_hdbscan_analysis['zone_name'].isin(['unassigned', 'ERROR_SIN_BAUTIZO'])
].copy()

# --- 2. REPORTE DE LIMPIEZA ---
puntos_originales = len(df_hdbscan_analysis)
puntos_limpios = len(df_kepler_clean)
puntos_removidos = puntos_originales - puntos_limpios

print(f"🧹 Se barrieron {puntos_removidos} puntos de ruido (Outliers/Unassigned).")
print(f"🗺️ Puntos estratégicos puros listos para mapear: {puntos_limpios}")

# --- 3. EXPORTACIÓN FINAL (FINAL) ---
# Guardamos este archivo específico para visualización
OUTPUT_KEPLER_CLEAN = '/content/drive/MyDrive/_Pienza/Assets/Phase_2/hdbscan_bautizados_kepler_clean_streamlit.csv'
df_kepler_clean.to_csv(OUTPUT_KEPLER_CLEAN, index=False)

display(Markdown(f"✨ **Archivo ultra-limpio para Kepler guardado en:** `{OUTPUT_KEPLER_CLEAN}`"))

### 🧹 **Operación Escoba: Limpiando el Mapa para Kepler.gl**

🧹 Se barrieron 2144 puntos de ruido (Outliers/Unassigned).
🗺️ Puntos estratégicos puros listos para mapear: 2616


✨ **Archivo ultra-limpio para Kepler guardado en:** `/content/drive/MyDrive/_Pienza/Assets/Phase_2/hdbscan_bautizados_kepler_clean_streamlit.csv`

In [50]:
# ==============================================================================
# CELL: VISUALIZACIÓN IN-NOTEBOOK (EL MAPA BAUTIZADO Y LIMPIO)
# ==============================================================================
import plotly.express as px
from IPython.display import display, Markdown

display(Markdown("### 🗺️ **Mapa Interactivo Opus: Territorios Bautizados (Sin Ruido)**"))

# 1. Preparar datos (usamos el dataset limpio)
# Filtramos al vuelo por si acaso, asegurando que no haya 'unassigned'
df_plot = df_hdbscan_analysis[~df_hdbscan_analysis['zone_name'].isin(['unassigned', 'ERROR_SIN_BAUTIZO'])].copy()

# 2. Ordenar alfabéticamente para que la leyenda sea fácil de navegar
df_plot.sort_values(by='zone_name', inplace=True)

# 3. Generar el Mapa en Plotly
fig_inline = px.scatter_mapbox(
    df_plot,
    lat="lat_viz",      # Usamos las coordenadas con Jitter para revelar densidad real
    lon="lon_viz",

    # --- LA MAGIA SEMÁNTICA ---
    color="zone_name",
    hover_name="zone_name",
    # --------------------------

    hover_data={
        "hdbscan_cluster_id": True, # Mantenemos el ID numérico en el tooltip por si acaso
        "lat_viz": False,
        "lon_viz": False,
    },
    mapbox_style="carto-darkmatter", # Fondo oscuro y sobrio (estilo Pienza)
    zoom=10.5,
    height=850,
    title='Territorios Estratégicos Opus (Consolidados y Bautizados)',

    # Usamos una paleta amplia porque tenemos 44 zonas
    color_discrete_sequence=px.colors.qualitative.Alphabet
)

# 4. Mostrar en pantalla
fig_inline.show()

### 🗺️ **Mapa Interactivo Opus: Territorios Bautizados (Sin Ruido)**

In [51]:
# ==============================================================================
# CELL: LA CUADRÍCULA SOBERANA (H3 HEXAGONS)
# ==============================================================================
!pip install h3 geopandas shapely

import h3
import pandas as pd
import geopandas as gpd
from shapely.geometry import Polygon
import plotly.express as px
from IPython.display import display, Markdown

# --- 0. CONFIGURACIÓN DEL EXPERIMENTO ---
H3_RESOLUTION = 10 # <-- ¡Cámbialo aquí para probar! (10 = grande, 12 = minúsculo)

display(Markdown(f"### 🛑 **Cuadrícula H3 (Resolución {H3_RESOLUTION})**"))

# 1. Usar el dataset limpio (sin ruido)
# Asumimos que df_kepler_clean se generó en la celda anterior
df_h3 = df_kepler_clean.copy()

# 2. Funciones de compatibilidad para H3 (Maneja versiones antiguas y nuevas de la librería)
def get_h3_index(lat, lon, res):
    try:
        return h3.latlng_to_cell(lat, lon, res) # H3 v4+
    except AttributeError:
        return h3.geo_to_h3(lat, lon, res)      # H3 v3

def get_h3_boundary(h3_id):
    try:
        coords = h3.cell_to_boundary(h3_id)     # H3 v4+
    except AttributeError:
        coords = h3.h3_to_geo_boundary(h3_id)   # H3 v3
    # Intercambiar (lat, lon) a (lon, lat) para Shapely/GeoJSON
    return Polygon([(lon, lat) for lat, lon in coords])

# 3. Mapear cada punto a su Hexágono
print(f"⏳ Convirtiendo {len(df_h3)} coordenadas a Hexágonos H3 (Res {H3_RESOLUTION})...")
df_h3['h3_index'] = df_h3.apply(lambda row: get_h3_index(row['dropoff_lat'], row['dropoff_lon'], H3_RESOLUTION), axis=1)

# 4. Agregación: Contar cuántos dropoffs hay por Hexágono y conservar su zona
df_h3_agg = df_h3.groupby(['h3_index', 'zone_name']).size().reset_index(name='offer_count')
print(f"✅ Agregación completa. Se generaron {len(df_h3_agg)} hexágonos únicos.")

# 5. EXPORTACIÓN PARA KEPLER.GL (LA VÍA RÁPIDA)
OUTPUT_H3_KEPLER = f'/content/drive/MyDrive/_Pienza/Assets/Phase_2/h3_res{H3_RESOLUTION}_clusters_streamlit.csv'
df_h3_agg.to_csv(OUTPUT_H3_KEPLER, index=False)
display(Markdown(f"💾 **Archivo H3 exportado a:** `{OUTPUT_H3_KEPLER}`"))

# 6. VISUALIZACIÓN EN LA LIBRETA (OPUS THEME)
print("⏳ Generando geometrías para visualización en Plotly...")
# Para Plotly necesitamos construir los polígonos
df_h3_agg['geometry'] = df_h3_agg['h3_index'].apply(get_h3_boundary)
gdf_h3 = gpd.GeoDataFrame(df_h3_agg, geometry='geometry', crs="EPSG:4326")

# Ordenamos para la leyenda
gdf_h3.sort_values(by='zone_name', inplace=True)

fig = px.choropleth_mapbox(
    gdf_h3,
    geojson=gdf_h3.geometry,
    locations=gdf_h3.index,
    color="zone_name",
    hover_name="zone_name",
    hover_data={"h3_index": True, "offer_count": True},
    mapbox_style="carto-darkmatter",
    center={"lat": 19.4288788, "lon": -99.1747491}, # Centrado en Anzures
    zoom=10.5,
    height=850,
    opacity=0.7,
    title=f'Sovereign Zones Cuantificadas (H3 Res {H3_RESOLUTION})',
    color_discrete_sequence=px.colors.qualitative.Alphabet
)

# Ajustes visuales para remover los bordes por defecto del choropleth si están muy gruesos
fig.update_traces(marker_line_width=0.5, marker_line_color='black')
fig.show()

Output hidden; open in https://colab.research.google.com to view.

In [52]:
# ==============================================================================
# CELL: EXPORTACIÓN H3 RES 9 - ESTRATEGIA "OCKHAM" (SOLO ZONAS SOBERANAS)
# ==============================================================================
import h3
import pandas as pd
from IPython.display import display, Markdown

# --- 1. PARÁMETROS ---
H3_RES_OCKHAM = 9
OUTPUT_PATH_H3 = f'/content/drive/MyDrive/_Pienza/Assets/Phase_2/h3_res{H3_RES_OCKHAM}_ockham_clean_streamlit.csv'

display(Markdown(f"### 🪒 **Ockham's Razor: Destilando H3 Res {H3_RES_OCKHAM}**"))

# --- 2. PROCESAMIENTO ---
# Usamos df_kepler_clean que YA FILTRÓ los 'unassigned' (Sovereign Filter Doctrine) [cite: 1830]
df_ockham = df_kepler_clean.copy()

def latlng_to_h3(row):
    # Compatibilidad con h3-py v3 y v4
    try:
        return h3.latlng_to_cell(row['dropoff_lat'], row['dropoff_lon'], H3_RES_OCKHAM)
    except AttributeError:
        return h3.geo_to_h3(row['dropoff_lat'], row['dropoff_lon'], H3_RES_OCKHAM)

print(f"⏳ Mapeando {len(df_ockham)} puntos a hexágonos Res {H3_RES_OCKHAM}...")
df_ockham['h3_index'] = df_ockham.apply(latlng_to_h3, axis=1)

# --- 3. AGREGACIÓN (EL CORAZÓN DEL ENTREGABLE) ---
# Agrupamos por hexágono Y zona para mantener el nombre y contar ofertas
df_h3_final = df_ockham.groupby(['h3_index', 'zone_name']).size().reset_index(name='offer_count')

# --- 4. EXPORTACIÓN ---
df_h3_final.to_csv(OUTPUT_PATH_H3, index=False)

# --- 5. REPORTE ---
print(f"✅ ¡FinalFinalV2 completado!")
print(f"📦 Hexágonos únicos generados: {len(df_h3_final)}")
print(f"🏆 Cobertura: {df_h3_final['zone_name'].nunique()} zonas soberanas representadas.")
display(Markdown(f"💾 **Archivo listo para Kepler/WebApp:** `{OUTPUT_PATH_H3}`"))

# Preview para tu tranquilidad
display(df_h3_final.sort_values(by='offer_count', ascending=False).head(10))

### 🪒 **Ockham's Razor: Destilando H3 Res 9**

⏳ Mapeando 2616 puntos a hexágonos Res 9...
✅ ¡FinalFinalV2 completado!
📦 Hexágonos únicos generados: 525
🏆 Cobertura: 44 zonas soberanas representadas.


💾 **Archivo listo para Kepler/WebApp:** `/content/drive/MyDrive/_Pienza/Assets/Phase_2/h3_res9_ockham_clean_streamlit.csv`

,h3_index,zone_name,offer_count
331,894995b9453ffff,terminal_1_aicm,99
522,894995bb263ffff,terminal_2_aicm,80
512,894995baea7ffff,carso_antara_miyana,45
243,894995b15c3ffff,santa_fe_patio,40
224,894995b1543ffff,nodo_constituyentes_reforma,38
278,894995b3367ffff,santa_fe_core,35
523,894995bb27bffff,terminal_2_aicm,31
294,894995b33cfffff,santa_fe_itesm,30
280,894995b3373ffff,santa_fe_core,28
271,894995b330fffff,santa_fe_core,27


In [53]:
# ==============================================================================
# CELL: GENERACIÓN DE ETIQUETAS ÚNICAS PARA CLUSTERS 3D
# ==============================================================================
import pandas as pd
import numpy as np
from IPython.display import display, Markdown

# 1. Cargar el dataframe de puntos original (el que tiene lat/lon individuales)
# Usamos df_kepler_clean porque ya tiene los nombres bautizados y sin ruido
df_points = df_kepler_clean.copy()

# 2. Calcular el Centroide Ponderado por Cluster
# Esto asegura que la etiqueta se coloque donde hay más densidad de ofertas
cluster_labels_df = df_points.groupby('zone_name').agg({
    'dropoff_lat': 'mean',
    'dropoff_lon': 'mean',
    'offer_id': 'count' # Solo para referencia de volumen
}).reset_index()

# 3. (Opcional) Agregar un ID numérico corto para la "Topological Blueprint"
# Esto mapea tus 44-45 clusters bautizados a un número simple
cluster_labels_df = cluster_labels_df.sort_values(by='zone_name').reset_index(drop=True)
cluster_labels_df['cluster_generic_id'] = cluster_labels_df.index + 1

# 4. Exportar el archivo de etiquetas
OUTPUT_CLUSTER_LABELS = '/content/drive/MyDrive/_Pienza/Assets/Phase_2/cluster_3d_labels_streamlit.csv'
cluster_labels_df.to_csv(OUTPUT_CLUSTER_LABELS, index=False)

display(Markdown(f"✅ **Archivo de etiquetas únicas generado:** `{OUTPUT_CLUSTER_LABELS}`"))
display(cluster_labels_df[['cluster_generic_id', 'zone_name']].head())

✅ **Archivo de etiquetas únicas generado:** `/content/drive/MyDrive/_Pienza/Assets/Phase_2/cluster_3d_labels_streamlit.csv`

,cluster_generic_id,zone_name
0,1,anzures_reforma
1,2,barranca_del_muerto
2,3,campos_eliseos
3,4,carso_antara_miyana
4,5,central_del_norte


In [54]:
# ==============================================================================
# CELL: EXPORTACIÓN PARA MAPEO MANUAL (SENSORIAL)
# ==============================================================================
import pandas as pd
from IPython.display import display, Markdown

# 1. Preparar el DataFrame de trabajo
# Usamos el resultado del Ockhamazo (Res 9) que ya tiene conteos
# Asumiendo que df_h3_final tiene [h3_index, zone_name, offer_count]
df_work = df_h3_final.copy()

# 2. Obtener coordenadas de los índices H3 para el cálculo
def h3_to_coords(h3_id):
    try:
        return h3.cell_to_latlng(h3_id) # H3 v4+
    except AttributeError:
        return h3.h3_to_geo(h3_id)      # H3 v3

df_work['coords'] = df_work['h3_index'].apply(h3_to_coords)
df_work['lat'] = df_work['coords'].apply(lambda x: x[0])
df_work['lon'] = df_work['coords'].apply(lambda x: x[1])

# 3. Calcular el Centro Ponderado (donde está el "peak" de cada zona)
df_work['w_lat'] = df_work['lat'] * df_work['offer_count']
df_work['w_lon'] = df_work['lon'] * df_work['offer_count']

mapping_session = df_work.groupby('zone_name').agg({
    'w_lat': 'sum',
    'w_lon': 'sum',
    'offer_count': 'sum'
}).reset_index()

mapping_session['lat_anchor'] = mapping_session['w_lat'] / mapping_session['offer_count']
mapping_session['lon_anchor'] = mapping_session['w_lon'] / mapping_session['offer_count']

# 4. Crear columna vacía para tu cerebro/ojos
mapping_session['manual_id'] = "" # AQUÍ ESCRIBIRÁS TÚ
mapping_session['comentarios'] = ""

# 5. Exportar CSV de sesión
SESSION_PATH = '/content/drive/MyDrive/_Pienza/Assets/Phase_2/working_session_mapping_streamlit.csv'
mapping_session[['zone_name', 'manual_id', 'lat_anchor', 'lon_anchor', 'offer_count', 'comentarios']].to_csv(SESSION_PATH, index=False)

display(Markdown(f"### 🧠 **Sesión de Mapeo Manual Iniciada**"))
display(Markdown(f"El archivo ha sido creado en: `{SESSION_PATH}`"))
print("Instrucciones:")
print("1. Abre este CSV en Google Sheets/Excel.")
print("2. Mira tu mapa 'The One' en Kepler.")
print("3. Llena la columna 'manual_id' con el número o nombre que prefieras.")
print("4. Guarda y re-importa a Kepler como capa de etiquetas.")

# Preview de lo que vas a mapear
display(mapping_session[['zone_name', 'offer_count', 'lat_anchor', 'lon_anchor']].sort_values('offer_count', ascending=False).head(10))

### 🧠 **Sesión de Mapeo Manual Iniciada**

El archivo ha sido creado en: `/content/drive/MyDrive/_Pienza/Assets/Phase_2/working_session_mapping_streamlit.csv`

Instrucciones:
1. Abre este CSV en Google Sheets/Excel.
2. Mira tu mapa 'The One' en Kepler.
3. Llena la columna 'manual_id' con el número o nombre que prefieras.
4. Guarda y re-importa a Kepler como capa de etiquetas.


,zone_name,offer_count,lat_anchor,lon_anchor
33,santa_fe_core,331,19.364837,-99.266792
3,carso_antara_miyana,170,19.440358,-99.201661
39,terminal_1_aicm,111,19.435519,-99.072303
40,terminal_2_aicm,111,19.422027,-99.077881
36,sotelo_san_esteban,105,19.454417,-99.220924
13,el_semaforo_de_palmas,96,19.433031,-99.211443
42,vialidad_de_la_barranca,86,19.404158,-99.273735
22,nodo_constituyentes_reforma,85,19.387204,-99.251512
2,campos_eliseos,75,19.429177,-99.194007
27,polanco_trebol,74,19.438013,-99.183556


In [55]:
# ==============================================================================
# CELL: CANONICAL BAUTIZO + KEPLER DUMMY ID (OCULTO/VISUAL)
# ==============================================================================
import plotly.express as px
from IPython.display import display, Markdown

display(Markdown("### 🏛️ **Restaurando Soberanía: Canonical Names + Kepler Dummy IDs**"))

# --- 1. DICCIONARIO CANÓNICO (Original Cluster ID -> Name) ---
canonical_map = {
    -1: 'unassigned',
    0: 'viaducto_tlalpan',
    1: 'terminal_1_aicm',
    2: 'central_del_norte',
    3: 'terminal_2_aicm',
    4: 'terminal_2_aicm',
    5: 'plaza_satelite_mundo_e',
    6: 'lomas_verdes',
    7: 'san_jeronimo',
    8: 'pedregal',
    9: 'san_angel_altavista',
    10: 'observatorio',
    11: 'sotelo_san_esteban',
    12: 'barranca_del_muerto',
    13: 'haciendas_san_fernando',
    14: 'vialidad_de_la_barranca',
    15: 'interlomas_magnocentro',
    16: 'santa_fe_itesm',
    17: 'cuajimalpa_centro',
    18: 'santa_fe_patio',
    19: 'santa_fe_abc',
    20: 'tamarindos',
    21: 'nodo_constituyentes_reforma',
    22: 'felix_cuevas',
    23: 'cruce_echanove',
    24: 'santa_fe_core',
    25: 'pabellon_bosques',
    26: 'duraznos',
    27: 'chamizal',
    28: 'lomas_anahuac',
    29: 'centro_historico',
    30: 'napoles_wtc',
    31: 'san_miguel_chapultepec',
    32: 'tacubaya',
    33: 'cofre_de_perote',
    34: 'condesa',
    35: 'roma_norte',
    36: 'col_juarez',
    37: 'la_diana',
    38: 'anzures_reforma',
    39: 'polanco_trebol',
    40: 'carso_antara_miyana',
    41: 'el_semaforo_de_palmas',
    42: 'campos_eliseos',
    43: 'virreyes',
    44: 'fc_cuernavaca'
}

# --- 2. DICCIONARIO DUMMY (Name -> Kepler ID 1-44) ---
kepler_id_map = {
    'lomas_verdes': 1, 'plaza_satelite_mundo_e': 2, 'central_del_norte': 3,
    'terminal_1_aicm': 4, 'terminal_2_aicm': 5, 'centro_historico': 6,
    'sotelo_san_esteban': 7, 'carso_antara_miyana': 8, 'campos_eliseos': 9,
    'polanco_trebol': 10, 'anzures_reforma': 11, 'la_diana': 12,
    'col_juarez': 13, 'roma_norte': 14, 'condesa': 15, 'napoles_wtc': 16,
    'san_miguel_chapultepec': 17, 'tacubaya': 18, 'observatorio': 19,
    'fc_cuernavaca': 20, 'virreyes': 21, 'el_semaforo_de_palmas': 22,
    'cofre_de_perote': 23, 'duraznos': 24, 'tamarindos': 25,
    'nodo_constituyentes_reforma': 26, 'santa_fe_patio': 27,
    'santa_fe_core': 28, 'santa_fe_itesm': 29, 'cruce_echanove': 30,
    'santa_fe_abc': 31, 'cuajimalpa_centro': 32, 'pabellon_bosques': 33,
    'chamizal': 34, 'lomas_anahuac': 35, 'vialidad_de_la_barranca': 36,
    'interlomas_magnocentro': 37, 'haciendas_san_fernando': 38,
    'felix_cuevas': 39, 'barranca_del_muerto': 40, 'san_angel_altavista': 41,
    'san_jeronimo': 42, 'pedregal': 43, 'viaducto_tlalpan': 44
}

# --- 3. PROCESAMIENTO ---
df_viz = df_hdbscan_analysis.copy()

# A. Restaurar Zone Name Canónico
df_viz['hdbscan_cluster_id'] = df_viz['hdbscan_cluster_id'].astype(int)
df_viz['zone_name'] = df_viz['hdbscan_cluster_id'].map(canonical_map)

# B. Crear Kepler Dummy ID basado en el nombre
df_viz['kepler_id'] = df_viz['zone_name'].map(kepler_id_map)

# C. LIMPIEZA TOTAL: Ojo con el -1 (Unassigned)
df_viz = df_viz[df_viz['hdbscan_cluster_id'] != -1]
df_viz = df_viz.dropna(subset=['kepler_id'])

# --- 4. MAPA DE VALIDACIÓN (DUMMY ID VISUAL) ---
fig = px.scatter_mapbox(
    df_viz,
    lat="lat_viz",
    lon="lon_viz",
    color="zone_name",
    hover_name="zone_name",
    # Mostramos tanto el ID Canónico como el Dummy en el hover para validar
    hover_data={"hdbscan_cluster_id": True, "kepler_id": True},
    mapbox_style="carto-positron",
    zoom=11,
    height=800,
    title='Validación: Zone Name (Canónico) con Kepler IDs (1-44)',
    color_discrete_sequence=px.colors.qualitative.Alphabet
)

fig.update_traces(marker=dict(size=7, opacity=0.8))
fig.show()

# --- 5. EXPORTACIÓN PARA KEPLER ---
OUTPUT_KEPLER = '/content/drive/MyDrive/_Pienza/Assets/Phase_2/hdbscan_kepler_final_streamlit.csv'
df_viz.to_csv(OUTPUT_KEPLER, index=False)
display(Markdown(f"✅ **Archivo listo para Kepler (con kepler_id):** `{OUTPUT_KEPLER}`"))

### 🏛️ **Restaurando Soberanía: Canonical Names + Kepler Dummy IDs**

✅ **Archivo listo para Kepler (con kepler_id):** `/content/drive/MyDrive/_Pienza/Assets/Phase_2/hdbscan_kepler_final_streamlit.csv`

In [56]:
# ==============================================================================
# CELL: GENERADOR DE ETIQUETAS (FIX FINAL: TYPO CORREGIDO)
# ==============================================================================
import pandas as pd
import h3
from IPython.display import display, Markdown

# 1. DICCIONARIO DE REPARACIÓN (Mapping de nombres a Kepler IDs 1-44)
repair_map = {
    'lomas_verdes': 1, 'plaza_satelite_mundo_e': 2, 'central_del_norte': 3,
    'terminal_1_aicm': 4, 'terminal_2_aicm': 5, 'centro_historico': 6,
    'sotelo_san_esteban': 7, 'carso_antara_miyana': 8, 'campos_eliseos': 9,
    'polanco_trebol': 10, 'anzures_reforma': 11, 'la_diana': 12,
    'col_juarez': 13, 'roma_norte': 14, 'condesa': 15, 'napoles_wtc': 16,
    'san_miguel_chapultepec': 17, 'tacubaya': 18, 'observatorio': 19,
    'fc_cuernavaca': 20, 'virreyes': 21, 'el_semaforo_de_palmas': 22,
    'cofre_de_perote': 23, 'duraznos': 24, 'tamarindos': 25,
    'nodo_constituyentes_reforma': 26, 'santa_fe_patio': 27,
    'santa_fe_core': 28, 'santa_fe_itesm': 29, 'cruce_echanove': 30,
    'santa_fe_abc': 31, 'cuajimalpa_centro': 32, 'pabellon_bosques': 33,
    'chamizal': 34, 'lomas_anahuac': 35, 'vialidad_de_la_barranca': 36,
    'interlomas_magnocentro': 37, 'haciendas_san_fernando': 38,
    'felix_cuevas': 39, 'barranca_del_muerto': 40, 'san_angel_altavista': 41,
    'san_jeronimo_tizapan': 43, 'pedregal': 42, 'viaducto_tlalpan': 44
}

# 2. IDENTIFICAR FUENTE DE DATOS
source_df = None
if 'df_h3_sandbox' in locals():
    source_df = df_h3_sandbox
elif 'df_h3_final' in locals():
    source_df = df_h3_final

if source_df is not None:
    df_labels = source_df.copy()

    # 3. REINJECTAR COLUMNAS FALTANTES (kepler_id y coordenadas)
    if 'kepler_id' not in df_labels.columns:
        df_labels['kepler_id'] = df_labels['zone_name'].map(repair_map)

    def h3_to_coords(h3_id):
        try: return h3.cell_to_latlng(h3_id)
        except: return h3.h3_to_geo(h3_id)

    df_labels['coords'] = df_labels['h3_index'].apply(h3_to_coords)
    df_labels['lat'] = df_labels['coords'].apply(lambda x: x[0])
    df_labels['lon'] = df_labels['coords'].apply(lambda x: x[1])

    # 4. CÁLCULO DE CENTROIDE PONDERADO (AQUÍ ESTABA EL ERROR)
    # Multiplicamos lat/lon por el volumen de ofertas para encontrar el "epicentro"
    df_labels['w_lat'] = df_labels['lat'] * df_labels['offer_count']
    df_labels['w_lon'] = df_labels['lon'] * df_labels['offer_count']

    labels_final = df_labels.groupby(['kepler_id', 'zone_name']).agg({
        'w_lat': 'sum',
        'w_lon': 'sum',
        'offer_count': 'sum'
    }).reset_index()

    # Dividimos entre el total de ofertas para normalizar la coordenada
    labels_final['lat_center'] = labels_final['w_lat'] / labels_final['offer_count']
    labels_final['lon_center'] = labels_final['w_lon'] / labels_final['offer_count']

    # 5. EXPORTACIÓN
    LABEL_PATH = '/content/drive/MyDrive/_Pienza/Assets/Phase_2/solo_etiquetas_opus_streamlit.csv'
    labels_final[['kepler_id', 'zone_name', 'lat_center', 'lon_center']].to_csv(LABEL_PATH, index=False)

    display(Markdown(f"### ✅ **Etiquetas Únicas Generadas**"))
    display(Markdown(f"El archivo con 44 etiquetas está listo en: `{LABEL_PATH}`"))
    display(labels_final[['kepler_id', 'zone_name', 'offer_count']].sort_values('kepler_id').head())
else:
    print("❌ Error: No se detectó ninguna tabla de hexágonos en memoria. Corre la celda de Sandbox primero.")

### ✅ **Etiquetas Únicas Generadas**

El archivo con 44 etiquetas está listo en: `/content/drive/MyDrive/_Pienza/Assets/Phase_2/solo_etiquetas_opus_streamlit.csv`

,kepler_id,zone_name,offer_count
0,1,lomas_verdes,45
1,2,plaza_satelite_mundo_e,32
2,3,central_del_norte,72
3,4,terminal_1_aicm,111
4,5,terminal_2_aicm,111


In [57]:
# ==============================================================================
# CELL: AUDITORÍA DE IDENTIDAD CON FOLIUM (FIX 42/43)
# ==============================================================================
import folium
from folium import plugins

# 1. DEFINIR EL CENTRO DEL MAPA (CDMX Sur)
map_center = [labels_final['lat_center'].mean(), labels_final['lon_center'].mean()]
m = folium.Map(location=map_center, zoom_start=12, tiles='OpenStreetMap')

# 2. AGREGAR MARCADORES CON ETIQUETAS VISIBLES
for idx, row in labels_final.iterrows():
    # Color especial para los sospechosos (42 y 43)
    dot_color = 'red' if row['kepler_id'] in [42, 43] else 'blue'

    # Agregar el punto
    folium.CircleMarker(
        location=[row['lat_center'], row['lon_center']],
        radius=6,
        color=dot_color,
        fill=True,
        fill_color=dot_color,
        tooltip=f"ID: {row['kepler_id']} - {row['zone_name']}"
    ).add_to(m)

    # Agregar el NÚMERO flotando (DivIcon)
    folium.map.Marker(
        [row['lat_center'], row['lon_center']],
        icon=folium.DivIcon(
            icon_size=(150,36),
            icon_anchor=(7,20),
            html=f'<div style="font-size: 14pt; color: {dot_color}; font-weight: bold;">{row["kepler_id"]}</div>',
            )
        ).add_to(m)

# 3. MOSTRAR MAPA
display(Markdown("### 🔍 **Mapa de Auditoría: Verifica el 42 y 43 en ROJO**"))
m

### 🔍 **Mapa de Auditoría: Verifica el 42 y 43 en ROJO**

In [58]:
# ==============================================================================
# CELDA 0: PIENZA COMMAND CENTER - BOOTSTRAP (BIGQUERY READY)
# ==============================================================================
from google.colab import auth
from google.cloud import bigquery
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# 1. AUTENTICACIÓN SOSTENIBLE
print("🔐 Autenticando identidad en Google Cloud...")
auth.authenticate_user()
print("✅ Usuario autenticado.")

# 2. CONFIGURACIÓN DEL PROYECTO
PROJECT_ID   = '645009831643'
DATASET_CORE = 'pienza_mini'
DATASET_BIG  = 'pienza_big'

# 3. INICIALIZACIÓN DE CLIENTE (La puerta de entrada a tus datos)
print(f"📡 Conectando al ecosistema BigQuery (Project: {PROJECT_ID})...")
client = bigquery.Client(project=PROJECT_ID)
print("✅ Cliente BigQuery listo.")

# 4. VISUAL CANON (OPUS LAB THEME) - Preparado para tus gráficas
OPUS_PURPLE = '#440154'
OPUS_TEAL   = '#21918c'
OPUS_GREY   = '#FAFAFA'
OPUS_TEXT   = '#121212'

sns.set_theme(style="whitegrid")
plt.rcParams.update({
    'figure.facecolor': OPUS_GREY, 'axes.facecolor': OPUS_GREY,
    'text.color': OPUS_TEXT, 'axes.titlecolor': OPUS_PURPLE,
    'axes.titleweight': 'bold', 'font.family': 'sans-serif'
})

print("\n--- SYSTEM READY: BQ Ecosistema Operativo ---")

🔐 Autenticando identidad en Google Cloud...
✅ Usuario autenticado.
📡 Conectando al ecosistema BigQuery (Project: 645009831643)...
✅ Cliente BigQuery listo.

--- SYSTEM READY: BQ Ecosistema Operativo ---


In [61]:
# ==============================================================================
# CELL: THE LEAN GOLDEN MASTER (DROPOFFS - ERD EXACTO)
# ==============================================================================
import pandas as pd
import h3
from IPython.display import display, Markdown

display(Markdown("### 🥇 **Forjando la Golden Master Lean (Pure SQL)**"))

# 1. SQL QUERY: Explotando tu ERD al milímetro + Fix de 'Accepted' + Redondeo
query = f"""
SELECT
    o.offer_id,
    pc.category_name AS product_category,
    ROUND(o.upfront_fare, 2) AS upfront_fare, -- MATAMOS LOS DECIMALES AQUÍ
    -- FIX: Si la razón es NULL, significa que el viaje fue aceptado
    COALESCE(rp.reason_primary_description, 'NULL (Accepted)') AS rejection_reason,
    ROUND(ef.eph_operational, 2) AS eph_operational -- MATAMOS LOS DECIMALES AQUÍ
FROM `{PROJECT_ID}.{DATASET_CORE}.offers` o
LEFT JOIN `{PROJECT_ID}.{DATASET_CORE}.engineered_features` ef
    ON o.offer_id = ef.offer_id_fk
LEFT JOIN `{PROJECT_ID}.{DATASET_CORE}.product_category` pc
    ON o.product_category_fk = pc.product_category_id
LEFT JOIN `{PROJECT_ID}.{DATASET_CORE}.reason_primary` rp
    ON o.reason_primary_fk = rp.reason_primary_id
"""

print("⏳ Extrayendo datos desnormalizados directo de BigQuery...")
df_bq = client.query(query).to_dataframe() # Asegúrate de tener tu client definido

# 2. PREPARAR DATOS ESPACIALES (La poda)
df_spatial = df_viz.copy()

# Calculamos H3
def get_h3(row):
    try: return h3.latlng_to_cell(row['dropoff_lat'], row['dropoff_lon'], 9)
    except: return h3.geo_to_h3(row['dropoff_lat'], row['dropoff_lon'], 9)

df_spatial['dropoff_h3_hex_id'] = df_spatial.apply(get_h3, axis=1)

# Renombramos a tu esquema
df_spatial = df_spatial.rename(columns={
    'hdbscan_cluster_id': 'dropoff_hdbscan_id',
    'zone_name': 'dropoff_hdbscan_name'
})

# Poda espacial
df_spatial = df_spatial[['offer_id', 'dropoff_h3_hex_id', 'dropoff_hdbscan_id', 'dropoff_hdbscan_name']]

# 3. EL JOIN FINAL (Datos Operativos + Espaciales)
print("⏳ Realizando Join Espacial...")
gold_master_df = pd.merge(df_bq, df_spatial, on='offer_id', how='inner')

# ==============================================================================
# 4. INYECCIÓN DE VOLUMEN Y TRUCO DE RENDERIZADO KEPLER
# ==============================================================================
print("⏳ Calculando densidad y aplicando truco de opacidad...")

# A) Calculamos el volumen total por hexágono y lo pegamos a todos los viajes
counts = gold_master_df.groupby('dropoff_h3_hex_id').size().reset_index(name='offer_volume')
gold_master_df = gold_master_df.merge(counts, on='dropoff_h3_hex_id')

# B) EL TRUCO: Creamos una columna que Kepler usará SOLO para dibujar.
# .mask() borra el ID (lo vuelve NaN) si el hexágono ya apareció antes.
gold_master_df['render_h3_hex'] = gold_master_df['dropoff_h3_hex_id'].mask(
    gold_master_df.duplicated(subset=['dropoff_h3_hex_id'])
)

# 5. EXPORTACIÓN
OUTPUT_GOLD = '/content/drive/MyDrive/_Pienza/Assets/Phase_2/gold_master_dropoffs_lean3.csv'
gold_master_df.to_csv(OUTPUT_GOLD, index=False)

display(Markdown(f"🏆 **Golden Master (Lean & Normalized) lista:** `{OUTPUT_GOLD}`"))
print(f"Dimensiones finales: {gold_master_df.shape}")
display(gold_master_df.head(5))

### 🥇 **Forjando la Golden Master Lean (Pure SQL)**

⏳ Extrayendo datos desnormalizados directo de BigQuery...
⏳ Realizando Join Espacial...
⏳ Calculando densidad y aplicando truco de opacidad...


🏆 **Golden Master (Lean & Normalized) lista:** `/content/drive/MyDrive/_Pienza/Assets/Phase_2/gold_master_dropoffs_lean3.csv`

Dimensiones finales: (2616, 10)


,offer_id,product_category,upfront_fare,rejection_reason,eph_operational,dropoff_h3_hex_id,dropoff_hdbscan_id,dropoff_hdbscan_name,offer_volume,render_h3_hex
0,OF00006,uberx,68.36,dropoff_non_operational,241.27,894995a31d7ffff,6,lomas_verdes,4,894995a31d7ffff
1,OF00007,uberx,71.13,low_profitability,164.15,894995bac57ffff,44,fc_cuernavaca,21,894995bac57ffff
2,OF00009,uberx,120.45,dropoff_non_operational,195.32,894995a335bffff,5,plaza_satelite_mundo_e,3,894995a335bffff
3,OF00012,uberx,162.00,NULL (Accepted),NaN,894995b1547ffff,21,nodo_constituyentes_reforma,17,894995b1547ffff
4,OF00017,uberx,102.37,dropoff_non_operational,211.80,89499586b47ffff,8,pedregal,9,89499586b47ffff
